## 04 — Phase 2: Fine-Tune on Real Data

### 0. Setup

In [ ]:
# Core imports
import os
import shutil
import random
from pathlib import Path
from collections import Counter

import yaml

# ── Project root ──────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
print(f"Project root: {PROJECT_ROOT}")

# ── Paths (all under data/) ────────────────────────────────────────────────────
DATA_DIR       = PROJECT_ROOT / 'data'
FINETUNE_DIR   = DATA_DIR / 'noodles_finetune_dataset'
TRAINING_DIR   = DATA_DIR / 'noodles_training'

# ── Shared constants ──────────────────────────────────────────────────────────
PIECE_LABELS     = list('ABCDEFGHIJK')               # piece classes 0-10 (unchanged)
ALL_CLASS_NAMES  = PIECE_LABELS + ['board', 'hinge'] # class 11 (board) + 12 (hinge)
NUM_CLASSES      = len(ALL_CLASS_NAMES)              # 13

PIECE_COLORS = {
    'A': ('Yellow',      (0xF9, 0xD6, 0x5E)),
    'B': ('SkyBlue',     (0x08, 0xA7, 0xE8)),
    'C': ('DarkBlue',    (0x20, 0x6D, 0xD9)),
    'D': ('Green',       (0x1F, 0xA1, 0x5B)),
    'E': ('Red',         (0xEE, 0x39, 0x4F)),
    'F': ('Teal',        (0x85, 0xDA, 0xBB)),
    'G': ('Pink',        (0xEC, 0x71, 0xA8)),
    'H': ('Purple',      (0xC7, 0x78, 0xB9)),
    'I': ('Orange',      (0xFC, 0x69, 0x0C)),
    'J': ('DarkRed',     (0xB6, 0x30, 0x48)),
    'K': ('YellowGreen', (0x95, 0xD4, 0x50)),
}

# Verify Phase 1 weights and files are available
p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'best.pt'
finetune_yaml = FINETUNE_DIR / 'dataset.yaml'

print(f"\n  Phase 1 best: {p1_best} {'✅' if p1_best.exists() else '❌ not found'}")
print(f"  Fine-tune YAML: {finetune_yaml} {'✅' if finetune_yaml.exists() else '❌ not found'}")

### 1. Inspect Available Datasets

In [12]:
for dataset_name in ['noodles_seg_dataset', 'noodles_finetune_dataset']:
    base = DATA_DIR / dataset_name
    print(f"\n📁 {dataset_name}/")
    for split in ['train', 'val']:
        img_dir = base / 'images' / split
        lbl_dir = base / 'labels' / split
        n_imgs = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
        n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f"  {split}: {n_imgs} images, {n_lbls} labels")
    
    yaml_file = base / 'dataset.yaml'
    if yaml_file.exists():
        with open(yaml_file) as f:
            cfg = yaml.safe_load(f)
        print(f"  YAML: nc={cfg.get('nc')}, names={list(cfg.get('names', {}).values())[:3]}...")
    else:
        print(f"  YAML: ❌ not found")


📁 noodles_seg_dataset/
  train: 3978 images, 1326 labels
  val: 975 images, 325 labels
  YAML: nc=None, names=['A_Yellow', 'B_SkyBlue', 'C_DarkBlue']...

📁 noodles_finetune_dataset/
  train: 84 images, 84 labels
  val: 20 images, 20 labels
  YAML: nc=11, names=['A_Yellow', 'B_SkyBlue', 'C_DarkBlue']...


### 2. Create Val Split from Train (if needed)

In [13]:
train_imgs = FINETUNE_DIR / 'images' / 'train'
val_imgs   = FINETUNE_DIR / 'images' / 'val'
train_lbls = FINETUNE_DIR / 'labels' / 'train'
val_lbls   = FINETUNE_DIR / 'labels' / 'val'

n_val = len(list(val_imgs.glob('*'))) if val_imgs.exists() else 0
n_train = len(list(train_imgs.glob('*'))) if train_imgs.exists() else 0

VAL_FRACTION = 0.2

if n_val == 0 and n_train > 0:
    print(f"No val set found. Splitting {VAL_FRACTION*100:.0f}% from train ({n_train} images)...")
    
    val_imgs.mkdir(parents=True, exist_ok=True)
    val_lbls.mkdir(parents=True, exist_ok=True)
    
    all_imgs = sorted(train_imgs.glob('*'))
    n_val_target = max(1, int(len(all_imgs) * VAL_FRACTION))
    val_selection = random.sample(all_imgs, n_val_target)
    
    for img_file in val_selection:
        shutil.move(str(img_file), str(val_imgs / img_file.name))
        lbl_src = train_lbls / (img_file.stem + '.txt')
        if lbl_src.exists():
            shutil.move(str(lbl_src), str(val_lbls / lbl_src.name))
    
    print(f"  Moved {len(val_selection)} images+labels to val")
    print(f"  Train: {len(list(train_imgs.glob('*')))} images remaining")
    print(f"  Val:   {len(list(val_imgs.glob('*')))} images")
else:
    print(f"Val set exists with {n_val} images. No split needed.")

Val set exists with 20 images. No split needed.


### 3. Fine-Tune (Phase 2)

In [ ]:
from ultralytics import YOLO

# ── Load Phase 1 best weights ─────────────────────────────────────────────────
p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'best.pt'
if not p1_best.exists():
    p1_best = TRAINING_DIR / 'phase1_synthetic' / 'weights' / 'last.pt'
assert p1_best.exists(), f"Phase 1 weights not found! Run notebook 02 first."

model = YOLO(str(p1_best))
print(f"✅ Loaded Phase 1 weights: {p1_best}")

# ── Fine-tune ─────────────────────────────────────────────────────────────────
# TODO: The real Roboflow dataset (noodles_finetune_dataset) currently contains only
# 11 piece classes (A-K). Classes 11 (board) and 12 (hinge) annotations must be
# added manually in Roboflow before they can benefit from real-data fine-tuning.
# Until then, Phase 2 updates piece classes only; board/hinge detection relies on
# the Phase 1 synthetic pre-training. The finetune dataset.yaml has been updated
# to nc=13 so Ultralytics does not raise a class-count mismatch when loading
# Phase 1 weights, but real images simply contain no board/hinge labels.
results_phase2 = model.train(
    data=str(finetune_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=15,
    freeze=5,
    lr0=0.001,
    save=True,
    save_period=10,
    project=str(TRAINING_DIR),
    name='phase2_finetune',
    exist_ok=True,
    # Lighter augmentation for real data
    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    mixup=0.05,
    copy_paste=0.1,
)

print("\n✅ Phase 2 fine-tuning complete!")

### 4. Validate & Compare

In [15]:
# Validate Phase 2 on finetune val set
metrics_p2 = model.val(
    data=str(finetune_yaml),
    imgsz=640,
    batch=16,
    device=0,
)

print("\n=== Phase 2 Validation Metrics (Real Data) ===")
print(f"  Box  mAP@0.5:     {metrics_p2.box.map50:.4f}")
print(f"  Box  mAP@0.5:0.95: {metrics_p2.box.map:.4f}")
print(f"  Mask mAP@0.5:     {metrics_p2.seg.map50:.4f}")
print(f"  Mask mAP@0.5:0.95: {metrics_p2.seg.map:.4f}")

# ── Compare with Phase 1 ──────────────────────────────────────────────────────
# Load Phase 1 results if available
p1_csv = TRAINING_DIR / 'phase1_synthetic' / 'results.csv'
if p1_csv.exists():
    import pandas as pd
    p1_results = pd.read_csv(p1_csv)
    # Get last epoch metrics
    last = p1_results.iloc[-1]
    # Column names may vary, look for map50 columns
    map50_cols = [c for c in p1_results.columns if 'map50' in c.lower() and 'map50-95' not in c.lower()]
    if map50_cols:
        p1_map50 = last[map50_cols[0]]
        print(f"\n=== Comparison ===")
        print(f"  Phase 1 mAP50: {p1_map50:.4f} (synthetic val)")
        print(f"  Phase 2 mAP50: {metrics_p2.seg.map50:.4f} (real val)")
else:
    print(f"\n⚠️  Phase 1 results.csv not found at {p1_csv}")

Ultralytics 8.4.21 🚀 Python-3.11.0rc1 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
YOLO26n-seg summary (fused): 139 layers, 2,691,029 parameters, 0 gradients, 9.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1447.4±968.7 MB/s, size: 30.9 KB)
val: Scanning /home/salumi/projects/project/data/noodles_finetune_dataset/labels/val.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 12.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0it/s 1.0s2.3s
                   all         20         68      0.921      0.935      0.965      0.942      0.921      0.935      0.965      0.897
              A_Yellow          4          4      0.941          1      0.995      0.995      0.941          1      0.995      0.968
             B_SkyBlue          7          7      0.834          1       0.96      0.936      0.834          1       0